---
# 1. BASELINE EVALUATION
---

In [ ]:
# ---------- 1. Install & imports ----------
# NOTE: Llama 3.1 is a gated model. HF login is required.
# Add your HF_TOKEN to Colab Secrets (key icon on left sidebar).
 
!pip install -q transformers accelerate bitsandbytes sentencepiece
 
import json, os, time, requests, pandas as pd, torch
from google.colab import drive, userdata
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed, BitsAndBytesConfig
from huggingface_hub import login

In [ ]:
# ---------- 2. HuggingFace login (required - Llama 3.1 is gated) ----------
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("HF login via secret successful.")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()
    print("Used manual login.")

In [ ]:
# ---------- 3. Mount Drive (for saving results only) ----------
drive.mount('/content/drive')

In [ ]:
# ---------- 4. Project paths ----------
# UPDATE THESE to match your GitHub repo and Drive folder
GITHUB_RAW    = "https://raw.githubusercontent.com/Heatbless/slm-safety-comparison/main/dataset"
 
# Drive is only used to save results
BASE_DIR      = "/content/drive/MyDrive/Colab Notebooks/slm_safety"
LOCAL_RESULTS = "/content/baseline_results.csv"
DRIVE_RESULTS = BASE_DIR + "/dataset/baseline_results.csv"
 
os.makedirs(BASE_DIR + "/dataset", exist_ok=True)
 
print(f"GitHub base : {GITHUB_RAW}")
print(f"Local temp  : {LOCAL_RESULTS}")
print(f"Drive final : {DRIVE_RESULTS}")

In [ ]:
# ---------- 5. Configuration ----------
MODEL_IDS = {
    "Qwen3-8B": "Qwen/Qwen3-8B",
    "Mistral-7B": "mistralai/Mistral-7B-Instruct-v0.2",
    "Vicuna-7B": "lmsys/vicuna-7b-v1.5",
    "Llama3.1-8B": "meta-llama/Meta-Llama-3.1-8B-Instruct"
}

PROMPT_FILE_NAMES = [
    "direct_injection.jsonl",
    "indirect_injection.jsonl",
    "persona_hijacking.jsonl",
    "hypothetical_framing.jsonl"
]

MAX_NEW_TOKENS = 256      # safe and fast
TEMPERATURE    = 0.0
DO_SAMPLE      = False
SEED           = 42

In [ ]:
# ---------- 6. GitHub JSONL loader ----------
def load_jsonl_from_github(base_url, filename):
    url = f"{base_url}/{filename}"
    response = requests.get(url)
    response.raise_for_status()
    return [
        json.loads(line)
        for line in response.text.strip().splitlines()
        if line.strip()
    ]

In [ ]:
# ---------- 7. Load adversarial prompts (from dataset/) ----------
all_prompts = []
for fname in PROMPT_FILE_NAMES:
    prompts = load_jsonl_from_github(GITHUB_RAW, fname)
    all_prompts.extend(prompts)
    print(f"Loaded {len(prompts)} prompts from {fname}")
 
prompt_df = pd.DataFrame(all_prompts)
print(f"\nTotal: {len(prompt_df)} adversarial prompts loaded from GitHub")

In [ ]:
# ---------- 8. 4-bit quantization config ----------
# NOTE: weights are stored in 4-bit NF4 but dequantized to BF16 for computation.
# This is why VRAM usage reflects BF16 levels during inference.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# ---------- 9. Resilient inference loop ----------
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

results = []
done = set()
if os.path.exists(LOCAL_RESULTS):
    results_df = pd.read_csv(LOCAL_RESULTS)
    results = results_df.to_dict(orient="records")
    for _, row in results_df.iterrows():
        done.add((row["model"], row["prompt_id"]))
    print(f"Resumed from local save. {len(results)} results loaded.")
elif os.path.exists(DRIVE_RESULTS):
    results_df = pd.read_csv(DRIVE_RESULTS)
    results = results_df.to_dict(orient="records")
    for _, row in results_df.iterrows():
        done.add((row["model"], row["prompt_id"]))
    print(f"Resumed from Drive save. {len(results)} results loaded.")
else:
    print("Starting fresh evaluation.")
 
total_cells = len(MODEL_IDS) * len(prompt_df)
remaining   = total_cells - len(done)
print(f"Total prompts to evaluate: {total_cells} (remaining: {remaining})")
 
# Loop over models
for model_name, model_id in MODEL_IDS.items():
    print(f"\n{'='*60}")
    print(f"Processing model: {model_name} ({model_id})")
 
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
 
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    model.eval()
 
    for idx, row in prompt_df.iterrows():
        prompt_id = row["prompt_id"]
        if (model_name, prompt_id) in done:
            continue
 
        technique   = row["technique"]
        difficulty  = row["difficulty"]
        prompt_text = row["prompt"]
 
        messages = [{"role": "user", "content": prompt_text}]
 
        if "vicuna" in model_name.lower():
            prompt_manual = f"USER: {prompt_text}\nASSISTANT:"
            inputs = tokenizer(prompt_manual, return_tensors="pt").to(device)
        else:
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                enable_thinking=False,
                return_tensors="pt"
            ).to(device)
 
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                do_sample=DO_SAMPLE,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
 
        input_length  = inputs.input_ids.shape[1]
        generated_ids = outputs[0][input_length:]
        response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
 
        result = {
            "model"       : model_name,
            "prompt_id"   : prompt_id,
            "technique"   : technique,
            "difficulty"  : difficulty,
            "prompt"      : prompt_text,
            "response"    : response_text,
            "input_tokens" : input_length,
            "output_tokens": len(generated_ids),
        }
        results.append(result)
        done.add((model_name, prompt_id))
 
        pd.DataFrame(results).to_csv(LOCAL_RESULTS, index=False)
 
        if len(results) % 50 == 0:
            !cp "{LOCAL_RESULTS}" "{DRIVE_RESULTS}"
            print(f"   Synced to Drive at {len(results)} results")
 
        if len(results) % 20 == 0:
            print(f"   Saved {len(results)} results so far")
 
    del model
    torch.cuda.empty_cache()
    time.sleep(1)

In [ ]:
# ------------ 10. Final sync to Drive ----------
!cp "{LOCAL_RESULTS}" "{DRIVE_RESULTS}"
print(f"\nBaseline evaluation complete. {len(results)} results saved.")